In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
!apt-get install sox


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
sox is already the newest version (14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [ ]:
!pip install soundfile pandas

In [ ]:
import os
import pandas as pd
import subprocess

# Paths
csv_path = "/content/drive/MyDrive/asr_audio/smalldata.csv"
output_dir = "/content/drive/MyDrive/asr_audio"  # Folder where the converted wav files will be saved
folders = {
    "/content/drive/MyDrive/asr_audio/fabm/signal": "fabm",  # Mapping folder paths to names for proper path construction
    "/content/drive/MyDrive/asr_audio/facs/signal": "facs",
}

# Step 1: Convert all .sph files to .wav in the respective folders
for folder, folder_name in folders.items():
    for filename in os.listdir(folder):
        if filename.endswith(".sph"):
            sph_path = os.path.join(folder, filename)
            # Construct the wav path based on the folder where the files are stored
            wav_path = os.path.join(output_dir, folder_name, "signal", filename.replace(".sph", ".wav"))
            if not os.path.exists(wav_path):  # Avoid re-conversion
                subprocess.run(["sox", sph_path, wav_path])
                print(f"Converted: {filename} -> {os.path.basename(wav_path)}")

# Step 2: Update CSV 'audio' column paths from .sph to correct .wav path
df = pd.read_csv(csv_path)

# Replace the .sph with the correct .wav path in the 'audio' column
def update_audio_path(audio_path):
    # Check which folder the audio belongs to (fabm or facs) and adjust the path accordingly
    if 'fabm' in audio_path:
        audio_path = audio_path.replace('/content/extracted_data/cmu_kids/kids/fabm/signal', '/content/drive/MyDrive/asr_audio/fabm/signal').replace('.sph', '.wav')
    elif 'facs' in audio_path:
        audio_path = audio_path.replace('/content/extracted_data/cmu_kids/kids/facs/signal', '/content/drive/MyDrive/asr_audio/facs/signal').replace('.sph', '.wav')

    # Ensure 'signal' is only in the right place
    audio_path = audio_path.replace("signal/signal", "signal")
    return audio_path

# Apply the update to the 'audio' column
df['audio'] = df['audio'].apply(update_audio_path)

# Save the updated CSV back to the same file (smalldata_wav.csv)
updated_csv_path = "/content/drive/MyDrive/asr_audio/smalldata_wav.csv"
df.to_csv(updated_csv_path, index=False)

print(f"\n✅ Updated CSV saved to: {updated_csv_path}")



✅ Updated CSV saved to: /content/drive/MyDrive/asr_audio/smalldata_wav.csv


In [ ]:
import pandas as pd

# Load the updated CSV file
df = pd.read_csv("/content/drive/MyDrive/asr_audio/smalldata_wav.csv")

# Copy the 'audio' column into a new 'path' column
df['path'] = df['audio']

# Save the updated CSV in the same file
df.to_csv("/content/drive/MyDrive/asr_audio/smalldata_wav.csv", index=False)

print("\n✅ 'path' column added and updated in the same smalldata_wav.csv file.")



✅ 'path' column added and updated in the same smalldata_wav.csv file.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your data from the CSV file
df = pd.read_csv('/content/drive/MyDrive/asr_audio/smalldata_wav.csv')

# Split the data into training (80%) and testing (20%) datasets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Save the split data into separate CSV files inside the 'asr_audio' folder
train_df.to_csv('/content/drive/MyDrive/asr_audio/train.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/asr_audio/test.csv', index=False)

print(f"Training data saved to: /content/drive/MyDrive/asr_audio/train.csv")
print(f"Testing data saved to: /content/drive/MyDrive/asr_audio/test.csv")



Training data saved to: /content/drive/MyDrive/asr_audio/train.csv
Testing data saved to: /content/drive/MyDrive/asr_audio/test.csv


In [ ]:
from datasets import load_dataset, DatasetDict

# Load the train and test splits from the CSV files correctly
common_voice = DatasetDict({
    "train": load_dataset(
        "csv",
        data_files="/content/drive/MyDrive/asr_audio/train.csv",
        delimiter=",",
        split="train"  # this avoids nested DatasetDict
    ),
    "test": load_dataset(
        "csv",
        data_files="/content/drive/MyDrive/asr_audio/test.csv",
        delimiter=",",
        split="train"  # same here
    )
})

# Now the structure is flat
print(common_voice)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'sentence', 'path'],
        num_rows: 85
    })
    test: Dataset({
        features: ['audio', 'sentence', 'path'],
        num_rows: 22
    })
})


In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")

In [ ]:
from datasets import load_dataset, DatasetDict

common_voice = DatasetDict({
    "train": load_dataset("csv", data_files="/content/drive/MyDrive/asr_audio/train.csv", split="train"),
    "test": load_dataset("csv", data_files="/content/drive/MyDrive/asr_audio/test.csv", split="train"),
})


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

# Initialize the tokenizer (Example: for the Whisper model or other ASR models)
tokenizer = AutoTokenizer.from_pretrained("openai/whisper-large")

# Extract the first sentence from your 'train' split
input_str = common_voice["train"][0]["sentence"]

# Tokenize the sentence
labels = tokenizer(input_str).input_ids

# Decode the tokens with special tokens (e.g., for Whisper)
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)

# Decode the tokens without special tokens
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

# Print the results
print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


Input:                 volcanoes m ey d some high mountains there too
Decoded w/ special:    <|startoftranscript|><|notimestamps|>volcanoes m ey d some high mountains there too<|endoftext|>
Decoded w/out special: volcanoes m ey d some high mountains there too
Are equal:             True


In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="English", task="transcribe")

In [ ]:
print(common_voice["train"][0])

{'audio': '/content/drive/MyDrive/asr_audio/facs/signal/facs2co2.wav', 'sentence': 'volcanoes m ey d some high mountains there too', 'path': '/content/drive/MyDrive/asr_audio/facs/signal/facs2co2.wav'}


In [ ]:
from datasets import Audio

common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))


In [ ]:
print(common_voice["train"][0])


{'audio': {'path': '/content/drive/MyDrive/asr_audio/facs/signal/facs2co2.wav', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00048828,
       -0.00048828, -0.00048828]), 'sampling_rate': 16000}, 'sentence': 'volcanoes m ey d some high mountains there too', 'path': '/content/drive/MyDrive/asr_audio/facs/signal/facs2co2.wav'}


In [ ]:
def prepare_dataset(batch):
    # load and resample audio data from 48 to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # encode target text to label ids
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch


In [ ]:
common_voice = common_voice.map(prepare_dataset, num_proc=4)


Map (num_proc=4):   0%|          | 0/85 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/22 [00:00<?, ? examples/s]

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:
# model.generation_config.language = "english"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None


In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


In [ ]:
import evaluate

metric = evaluate.load("wer")


In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
	output_dir="omi./whisper-small-hi",  # change to a repo name of your choice
	per_device_train_batch_size=8,
	gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
	learning_rate=1e-4,
	warmup_steps=10,
	max_steps=200,
	gradient_checkpointing=True,
	fp16=True,
	eval_strategy="steps",
	per_device_eval_batch_size=8,
	predict_with_generate=True,
	generation_max_length=200,
	save_steps=50,
	eval_steps=50,
	logging_steps=5,
	report_to=["tensorboard"],
	load_best_model_at_end=True,
	metric_for_best_model="wer",
	greater_is_better=False,
	push_to_hub=False,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)


<ipython-input-72-69786f5d74d5>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
`use_cache = True` is incompatible with gradient checkpointing. Setting `use_cache = False`...


Step,Training Loss,Validation Loss,Wer
50,0.042400,1.021251,25.365854
100,0.064100,1.049543,22.439024
150,0.005300,1.110067,23.414634
200,0.000100,1.044033,22.439024


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 2

TrainOutput(global_step=200, training_loss=0.46317039037065116, metrics={'train_runtime': 573.6036, 'train_samples_per_second': 2.789, 'train_steps_per_second': 0.349, 'total_flos': 4.4615302889472e+17, 'train_loss': 0.46317039037065116, 'epoch': 18.181818181818183})